# Exercise 21.3: Causal Masking & Multi-Head Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-21/03-causal-masking-and-multihead.ipynb)

- **Book Reference:** Chapter 21 (*Mathematics of Transformers and Large Language Models*, Sections 21.2–21.3)
- **Estimated Time:** 20–25 minutes
- **Learning Objectives:**
  1. Construct an upper-triangular causal mask so token $t$ cannot attend to future positions $j > t$.
  2. Reshape $(T, d_{\text{model}})$ tensors into $(H, T, d_{\text{head}})$ and back for multi-head attention.
  3. Verify that causal attention weights are strictly $0$ above the main diagonal.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

## Mathematical Background

During autoregressive generation (Chapter 21 *Common Mistake*), token $i$ may only attend to tokens $j \le i$. We enforce this by adding a **causal mask** $M \in \mathbb{R}^{T \times T}$ before `softmax`:
$$M_{ij} = \begin{cases} 0 & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

In **Multi-Head Attention** with $H$ heads and $d_{\text{head}} = d_{\text{model}} / H$, we split $Q, K, V \in \mathbb{R}^{T \times d_{\text{model}}}$ into shape $(H, T, d_{\text{head}})$, run attention per head in parallel, and concatenate back to $(T, d_{\text{model}})$.

In [ ]:
def create_causal_mask(seq_len: int) -> np.ndarray:
    """Create an upper-triangular additive causal mask of shape (seq_len, seq_len).

    Entries with j > i are -np.inf; entries with j <= i are 0.0.
    """
    # TODO: Use np.triu with k=1 to place -np.inf above the main diagonal
    raise NotImplementedError("Complete this TODO")


def split_heads(X: np.ndarray, num_heads: int) -> np.ndarray:
    """Reshape (T, d_model) -> (num_heads, T, d_head)."""
    # TODO: Reshape (T, num_heads, d_head) then transpose to (num_heads, T, d_head)
    raise NotImplementedError("Complete this TODO")


def combine_heads(X_heads: np.ndarray) -> np.ndarray:
    """Reshape (num_heads, T, d_head) -> (T, num_heads * d_head)."""
    # TODO: Transpose to (T, num_heads, d_head) then reshape to (T, d_model)
    raise NotImplementedError("Complete this TODO")


def multi_head_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    num_heads: int,
    causal: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute multi-head scaled dot-product attention.

    Args:
        Q: Query matrix of shape (T, d_model).
        K: Key matrix of shape (T, d_model).
        V: Value matrix of shape (T, d_model).
        num_heads: Number of attention heads H (must divide d_model).
        causal: If True, apply additive causal mask.

    Returns:
        Tuple (output, weights) where output has shape (T, d_model)
        and weights has shape (num_heads, T, T).
    """
    # TODO: Split heads, compute scaled dot-product scores (+ causal mask if causal), softmax, @ V_h, and combine heads
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- `np.triu(np.ones((T, T), dtype=bool), k=1)` selects strictly upper-triangular entries (`j > i`).
- Do NOT fill the upper triangle with `-np.inf` via `0.0 * np.inf` (which produces `NaN`); assign `mask[upper] = -np.inf` directly.
- `combine_heads(split_heads(X, H))` should recover `X` identically.

</details>

In [ ]:
T, d_model, H = 6, 12, 3
X = rng.normal(size=(T, d_model))
assert np.allclose(combine_heads(split_heads(X, H)), X)

Q = rng.normal(size=(T, d_model))
K = rng.normal(size=(T, d_model))
V = rng.normal(size=(T, d_model))

out_causal, w_causal = multi_head_attention(Q, K, V, num_heads=H, causal=True)
assert out_causal.shape == (T, d_model)
assert w_causal.shape == (H, T, T)
assert np.allclose(w_causal.sum(axis=-1), 1.0)

# Strictly upper-triangular attention weights must be 0.0
upper_idx = np.triu_indices(T, k=1)
for h in range(H):
    assert np.allclose(w_causal[h][upper_idx], 0.0), f"Head {h} leaked future weights!"

print("All checks passed! Multi-head causal attention shape:", out_causal.shape)

## Think Deeper

1. Why must causal masking be applied *before* `softmax` rather than zeroing out upper-triangular entries *after* `softmax`?
2. Why do Transformers use multiple smaller heads ($H \times d_{\text{head}}$) instead of a single head of dimension $d_{\text{model}}$?